# Spark Large-Scale Analysis

Use the Spark profile to correlate large event windows without pulling the full dataset into a notebook kernel.

In [ ]:
import os
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from clario360 import Client

assert os.environ.get('SPARK_ENABLED') == 'true', 'This notebook requires the Spark Connected profile.'
jdbc_jar = os.environ.get('CLICKHOUSE_JDBC_JAR', '/opt/clario360/jars/clickhouse-jdbc-0.9.7-all.jar')
if not Path(jdbc_jar).exists():
    raise FileNotFoundError(f'ClickHouse JDBC jar not found at {jdbc_jar}. Rebuild the Spark notebook image or set CLICKHOUSE_JDBC_JAR correctly.')
sdk = Client.from_env()
spark = SparkSession.builder \
    .appName('Clario360SecurityAnalysis') \
    .master(os.environ['SPARK_MASTER']) \
    .config('spark.driver.host', os.environ.get('SPARK_DRIVER_HOST', os.environ.get('POD_IP', '127.0.0.1'))) \
    .config('spark.driver.bindAddress', os.environ.get('SPARK_DRIVER_BIND_ADDRESS', '0.0.0.0')) \
    .config('spark.driver.port', os.environ.get('SPARK_DRIVER_PORT', '29413')) \
    .config('spark.blockManager.port', os.environ.get('SPARK_BLOCKMANAGER_PORT', '29414')) \
    .config('spark.driver.memory', os.environ.get('SPARK_DRIVER_MEMORY', '4g')) \
    .config('spark.executor.memory', os.environ.get('SPARK_EXECUTOR_MEMORY', '4g')) \
    .config('spark.executor.cores', os.environ.get('SPARK_EXECUTOR_CORES', '2')) \
    .config('spark.jars', jdbc_jar) \
    .config('spark.driver.extraClassPath', jdbc_jar) \
    .config('spark.executor.extraClassPath', jdbc_jar) \
    .getOrCreate()
sdk.record_spark_job('started', 'Initialized Spark session', {'app_name': 'Clario360SecurityAnalysis', 'jdbc_jar': jdbc_jar})
print(spark.sparkContext.applicationId)

## Correlate failed and successful logins

This query pattern is useful for brute-force investigations over longer windows.

In [ ]:
jdbc_url = f"jdbc:clickhouse://{os.environ['CLICKHOUSE_HOST']}:{os.environ.get('CLICKHOUSE_PORT', '9000')}"
events_df = spark.read \
    .format('jdbc') \
    .option('url', jdbc_url) \
    .option('driver', 'com.clickhouse.jdbc.ClickHouseDriver') \
    .option('dbtable', '(SELECT * FROM security_events WHERE timestamp >= now() - INTERVAL 30 DAY) AS events') \
    .option('user', open('/etc/clario360/credentials/clickhouse_user').read().strip()) \
    .option('password', open('/etc/clario360/credentials/clickhouse_password').read().strip()) \
    .load()
failed = events_df.filter(F.col('action') == 'login_failure').select('source_ip', 'timestamp', 'user_id')
success = events_df.filter(F.col('action') == 'login_success').select('source_ip', F.col('timestamp').alias('success_time'), F.col('user_id').alias('target_user'))
correlated = failed.join(success, on='source_ip').filter((F.col('success_time') > F.col('timestamp')) & (F.col('success_time') < F.col('timestamp') + F.expr("INTERVAL 30 MINUTE"))).groupBy('source_ip', 'target_user').agg(F.count('*').alias('failed_attempts')).filter(F.col('failed_attempts') >= 5)
sdk.record_data_query('clickhouse', 'Spark JDBC read for brute-force correlation', {'window_days': 30})
correlated.show(10, truncate=False)

## Clean up

Stop the Spark session once the analysis is complete to release cluster resources.

In [ ]:
sdk.record_spark_job('completed', 'Spark correlation query completed', {'app_id': spark.sparkContext.applicationId})
spark.stop()